In [1]:
import os
os.chdir("D:/Projects/volatility-radar")

In [2]:
import pandas as pd
import numpy as np

price_df = pd.read_csv('data/processed_v2/pair_features.csv')
cal_df = pd.read_csv('data/processed_v2/calendar_features.csv')

print('Price shape:', price_df.shape)
print('Calendar shape:', cal_df.shape)

Price shape: (4038, 13)
Calendar shape: (13718, 16)


In [3]:
price_df.columns.to_list()

['pair',
 'date',
 'open',
 'high',
 'low',
 'close',
 'max_up_pips',
 'max_down_pips',
 'max_profit',
 'max_loss',
 'daily_return',
 'rolling_std',
 'label']

In [4]:
cal_df.columns.to_list()

['date',
 'time',
 'currency',
 'event',
 'impact',
 'actual',
 'previous',
 'actual_clean',
 'previous_clean',
 'change',
 'change_rel',
 'impact_num',
 'z_score',
 'surprise_z',
 'signal',
 'signal_surprise']

In [5]:
currency_map = {
    'EUR': ['EURUSD'],
    'GBP': ['GBPUSD'],
    'JPY': ['USDJPY'],
    'USD': ['EURUSD', 'GBPUSD', 'USDJPY']
}

rows = []
for _, row in cal_df.iterrows():
    for pair in currency_map[row['currency']]:
        new_row = row.copy()
        new_row['pair'] = pair
        rows.append(new_row)

cal_expanded = pd.DataFrame(rows).reset_index(drop=True)

print('Before expansion:', cal_df.shape)
print('After expansion:', cal_expanded.shape)
print(cal_expanded['pair'].value_counts())

Before expansion: (13718, 16)
After expansion: (23602, 17)
pair
EURUSD    9252
GBPUSD    7513
USDJPY    6837
Name: count, dtype: int64


In [14]:
cal_agg = cal_expanded.groupby(['date', 'pair']).agg(
    high_impact_count=('impact_num', lambda x: (x == 3).sum()),
    medium_impact_count=('impact_num', lambda x: (x == 2).sum()),
    low_impact_count=('impact_num', lambda x: (x == 1).sum()),
    max_z_score=('z_score', lambda x: x.abs().max()),
    sum_signal=('signal', 'sum'),
    dominant_direction=('z_score', 'mean'),
    max_surprise_z=('surprise_z', lambda x: x.abs().max()),
    sum_signal_surprise=('signal_surprise', 'sum')
).reset_index()

In [15]:
cal_agg.shape

(3968, 10)

In [16]:
cal_agg.head()

,date,pair,high_impact_count,medium_impact_count,low_impact_count,max_z_score,sum_signal,dominant_direction,max_surprise_z,sum_signal_surprise
0,2021-01-04,EURUSD,0,0,7,1.022003,0.186100,0.026586,1.505369,-0.387363
1,2021-01-04,GBPUSD,0,1,5,1.169623,1.849037,0.294428,1.169623,1.559278
2,2021-01-04,USDJPY,0,0,3,0.967588,1.289071,0.429690,0.877492,1.197206
3,2021-01-05,EURUSD,0,1,7,1.989072,6.627636,0.579820,1.989072,6.627636
4,2021-01-05,GBPUSD,0,1,2,1.989072,6.613944,1.541624,1.989072,6.613944


In [17]:
df = price_df.merge(cal_agg, on=['date', 'pair'], how='left')

In [20]:
cal_cols = ['high_impact_count', 'medium_impact_count', 'low_impact_count',
            'max_z_score', 'sum_signal', 'dominant_direction',
            'max_surprise_z', 'sum_signal_surprise']

df[cal_cols] = df[cal_cols].fillna(0)